# A Comparative Methodology Recommendation System for Rainfall Prediction Using Research Paper Analysis

## Project Objective
This project demonstrates a research-oriented machine learning framework:
1. **Adapts four published methodologies** on a common NASA POWER meteorological dataset (Hyderabad 2021–2026).
2. **Evaluates performance** under a rigorous common experimental protocol (chronological split, strict prevention of target leakage).
3. **Analyzes dataset characteristics** (sample size, dimensionality, missingness, outliers, skewness, autocorrelation, seasonality).
4. **Builds a structured methodology knowledge base** (`methodology_profiles.csv`).
5. **Recommends the single most suitable methodology** for any unseen rainfall dataset based on meta-feature affinity.
6. **Executes rainfall predictions** using only the recommended methodology.


In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Add project root to path
sys.path.insert(0, os.path.abspath('..'))

from src.data_loader import load_nasa_power_csv
from src.data_cleaning import clean_dataset
from src.dataset_profiler import profile_dataset
from src.methodology_recommender import MethodologyRecommender
from src.prediction import execute_recommended_prediction

print("Modules imported successfully!")


## 1. Load and Inspect Common Meteorological Dataset
We load the NASA POWER daily meteorological dataset for Hyderabad, Telangana, India (2021-01-01 to 2026-10-01).


In [ ]:
data_path = os.path.join('..', 'data', 'raw', 'POWER_Point_Daily_Hyderabad.csv')
df_raw, metadata = load_nasa_power_csv(data_path)
print(f"Loaded raw dataset shape: {df_raw.shape}")
print("Header metadata:", metadata)
df_raw.head()


## 2. Programmatic Inspection & Cleaning
We inspect NASA POWER fill values (`-999.0`), analyze `WD10M` (wind direction) anomalies, derive binary `Rain`, and strictly eliminate `PRECTOTCORR` to prevent target leakage.


In [ ]:
df_clean, cleaning_log, raw_prectotcorr = clean_dataset(df_raw, drop_unphysical_wd=True)
print("Decisions made during cleaning:")
for d in cleaning_log['decisions_made']:
    print(f" - {d}")
    
print("\nWD10M Diagnosis Summary:")
print(cleaning_log['wd10m_diagnosis'])


## 3. Dataset Profiling & Meta-Feature Extraction
Extract statistical, distributional, class balance, and time-series meta-features.


In [ ]:
profile, profile_text = profile_dataset(df_clean, target_col='Rain', date_col='Date', raw_rainfall_series=raw_prectotcorr)
print(profile_text)


## 4. Methodology Benchmark Comparison Results
Review the comparative metrics table generated from running the four published methodology adaptations.


In [ ]:
comparison_table_path = os.path.join('..', 'results', 'tables', 'methodology_comparison.csv')
if os.path.exists(comparison_table_path):
    df_results = pd.read_csv(comparison_table_path)
    display(df_results[['Methodology', 'Model', 'Accuracy', 'Rain Recall', 'Rain F1', 'ROC-AUC', 'Balanced Score']])


## 5. Methodology Knowledge Base
The knowledge base maps dataset characteristics to methodology suitability, strengths, and weaknesses.


In [ ]:
kb_path = os.path.join('..', 'methodology_profiles.csv')
if os.path.exists(kb_path):
    df_kb = pd.read_csv(kb_path)
    display(df_kb[['methodology', 'model_family', 'champion_model', 'affinity_temporal_order', 'strengths']])


## 6. Testing the Methodology Recommender on Unseen Dataset
We pass an unseen dataset (`data/test/unseen_holdout_dataset.csv`), profile its meta-features, and let the recommender select the single most suitable methodology.


In [ ]:
unseen_path = os.path.join('..', 'data', 'test', 'unseen_holdout_dataset.csv')
df_unseen_raw = pd.read_csv(unseen_path)
df_unseen_clean, _, _ = clean_dataset(df_unseen_raw, drop_unphysical_wd=True)

recommender = MethodologyRecommender(kb_path)
rec_result = recommender.recommend(df_unseen_clean)
print(rec_result['report_text'])


## 7. Rainfall Prediction Using the Recommended Methodology
Executing rainfall classification on the unseen data using ONLY the recommended methodology.


In [ ]:
pred_df, eval_metrics = execute_recommended_prediction(
    df_clean, df_unseen_clean, rec_result['recommended_methodology']
)
print("Predicted probabilities and actual classes:")
display(pred_df.head(10))

if eval_metrics:
    print("\nUnseen Evaluation Metrics:")
    for k, v in eval_metrics.items():
        if k != 'confusion_matrix':
            print(f"  {k:18s}: {v}")
